In [2]:
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Thu Feb  5 04:59:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q -U accelerate
!pip install -q -U peft
!pip install -q -U bitsandbytes
!pip install -q -U transformers
!pip install -q -U trl
!pip install -q -U datasets
!pip install -q -U wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 84.4 MB/s eta 0:00:00


In [4]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
import gc


In [5]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DATASET_NAME = "npv2k1/n8n-workflow"
OUTPUT_DIR = "./qwen-n8n-finetuned"
WANDB_PROJECT = "qwen-n8n-finetune"  # Tùy chọn: để track training

In [6]:
print("Loading dataset...")
dataset = load_dataset(DATASET_NAME)
print(dataset)

# Xem mẫu dữ liệu
print("\n=== Sample data ===")
print(dataset['train'][0])

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/63.0 [00:00<?, ?B/s]

dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1054 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 1054
    })
})

=== Sample data ===
{'input': 'Insert Excel data to Postgres', 'output': '{"nodes":[{"name":"Read Binary File","type":"n8n-nodes-base.readBinaryFile","position":[450,650],"parameters":{"filePath":"spreadsheet.xls"},"typeVersion":1},{"name":"Spreadsheet File1","type":"n8n-nodes-base.spreadsheetFile","position":[600,650],"parameters":{},"typeVersion":1},{"name":"Insert Rows1","type":"n8n-nodes-base.postgres","position":[750,650],"parameters":{"table":"product","columns":"name,ean"},"credentials":{"postgres":"postgres"},"typeVersion":1}],"connections":{"Read Binary File":{"main":[[{"node":"Spreadsheet File1","type":"main","index":0}]]},"Spreadsheet File1":{"main":[[{"node":"Insert Rows1","type":"main","index":0}]]}}}'}


In [7]:
def format_instruction(sample):
    """
    Format data thành prompt cho training
    Điều chỉnh function này dựa trên cấu trúc dataset thực tế
    """

    if 'instruction' in sample and 'output' in sample:
        instruction = sample['instruction']
        workflow = sample['output']
    elif 'prompt' in sample and 'completion' in sample:
        instruction = sample['prompt']
        workflow = sample['completion']
    else:
        # Fallback: lấy field đầu tiên làm instruction, field thứ 2 làm output
        keys = list(sample.keys())
        instruction = sample[keys[0]]
        workflow = sample[keys[1]]

    prompt = f"""<|im_start|>system
You are a helpful assistant that generates n8n workflow JSON based on user requirements.<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{workflow}<|im_end|>"""

    return prompt


In [8]:
print("\n=== Formatted prompt example ===")
print(format_instruction(dataset['train'][0]))


=== Formatted prompt example ===
<|im_start|>system
You are a helpful assistant that generates n8n workflow JSON based on user requirements.<|im_end|>
<|im_start|>user
Insert Excel data to Postgres<|im_end|>
<|im_start|>assistant
{"nodes":[{"name":"Read Binary File","type":"n8n-nodes-base.readBinaryFile","position":[450,650],"parameters":{"filePath":"spreadsheet.xls"},"typeVersion":1},{"name":"Spreadsheet File1","type":"n8n-nodes-base.spreadsheetFile","position":[600,650],"parameters":{},"typeVersion":1},{"name":"Insert Rows1","type":"n8n-nodes-base.postgres","position":[750,650],"parameters":{"table":"product","columns":"name,ean"},"credentials":{"postgres":"postgres"},"typeVersion":1}],"connections":{"Read Binary File":{"main":[[{"node":"Spreadsheet File1","type":"main","index":0}]]},"Spreadsheet File1":{"main":[[{"node":"Insert Rows1","type":"main","index":0}]]}}}<|im_end|>


In [9]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
# Cấu hình 4-bit quantization để tiết kiệm VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Cấu hình LoRA
peft_config = LoraConfig(
    r=16,  # Rank của LoRA
    lora_alpha=32,  # Alpha parameter
    lora_dropout=0.05,  # Dropout
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Qwen modules
)

In [11]:
# Load model với quantization
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Apply LoRA
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Nhỏ để fit vào T4
    gradient_accumulation_steps=4,  # Tăng effective batch size
    gradient_checkpointing=True,  # Tiết kiệm memory
    optim="paged_adamw_8bit",  # Optimizer tối ưu cho memory
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    bf16=False,  # T4 không support BF16 tốt
    fp16=False,  # Tắt FP16 để tránh lỗi với BFloat16
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",  # Đổi thành "wandb" nếu muốn dùng W&B
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
# Setup SFTTrainer
# TRL API version mới đơn giản hóa rất nhiều - chỉ cần 3 params cơ bản

# Format dataset trước thành text
print("Formatting dataset...")
def prepare_dataset(sample):
    formatted_text = format_instruction(sample)
    return {"text": formatted_text}

formatted_dataset = dataset['train'].map(prepare_dataset, remove_columns=dataset['train'].column_names)
print(f"Dataset formatted: {len(formatted_dataset)} samples")
print(f"Example: {formatted_dataset[0]['text'][:200]}...")

# Tạo trainer với 3 parameters cơ bản nhất
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args,
)

Formatting dataset...


Map:   0%|          | 0/1054 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Dataset formatted: 1054 samples
Example: <|im_start|>system
You are a helpful assistant that generates n8n workflow JSON based on user requirements.<|im_end|>
<|im_start|>user
Insert Excel data to Postgres<|im_end|>
<|im_start|>assistant
{"n...


Adding EOS to train dataset:   0%|          | 0/1054 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1054 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (369427 > 131072). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/1054 [00:00<?, ? examples/s]

In [14]:
# Train model
print("Starting training...")
trainer.train()

# Lưu model
print("Saving model...")
trainer.save_model(OUTPUT_DIR)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss
10,2.038814
20,1.563935
30,1.230928
40,1.185466
50,1.115004
60,1.102612
70,1.004929
80,1.010242
90,0.971787
100,1.025658


Saving model...


In [15]:
# Lưu model đã merge (optional - để inference dễ hơn)
print("Merging and saving final model...")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# Merge với LoRA weights
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model = model.merge_and_unload()

# Save merged model
merged_output_dir = f"{OUTPUT_DIR}-merged"
model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)
print(f"Merged model saved to {merged_output_dir}")

Merging and saving final model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to ./qwen-n8n-finetuned-merged


In [ ]:
# Test model
print("\n=== Testing model ===")

# Load model cho inference
pipe = pipeline(
    "text-generation",
    model=merged_output_dir,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
)

In [20]:
# Test prompt
test_prompt = """<|im_start|>system
You are a helpful assistant that generates n8n workflow JSON based on user requirements.<|im_end|>
<|im_start|>user
UNIX Timestamp Node<|im_end|>
<|im_start|>assistant
"""

# Generate
outputs = pipe(
    test_prompt,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.15,
)

print("\n=== Generated Workflow ===")
print(outputs[0]['generated_text'])

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Generated Workflow ===
<|im_start|>system
You are a helpful assistant that generates n8n workflow JSON based on user requirements.<|im_end|>
<|im_start|>user
UNIX Timestamp Node<|im_end|>
<|im_start|>assistant
{"id":"451","name":"Get UNIX timestamp from current date and time (in UTC)","type":"n8n-nodes-base.unixTimestamp","position":[270,640],"parameters":{"value":""},"credentials":{},"typeVersion":1}


In [21]:
# Test prompt
test_prompt = """<|im_start|>system
You are a helpful assistant that generates n8n workflow JSON based on user requirements.<|im_end|>
<|im_start|>user
Git backup of workflows and credentials<|im_end|>
<|im_start|>assistant
"""

# Generate
outputs = pipe(
    test_prompt,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.15,
)

print("\n=== Generated Workflow ===")
print(outputs[0]['generated_text'])

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Generated Workflow ===
<|im_start|>system
You are a helpful assistant that generates n8n workflow JSON based on user requirements.<|im_end|>
<|im_start|>user
Git backup of workflows and credentials<|im_end|>
<|im_start|>assistant
{"id":"734","name":"Git backup of workflows and credentials","nodes":[{"name":"Set","type":"n8n-nodes-base.set","position":[-160,290],"parameters":{"values":{"string":[{"key":"file_name","value":"={{$node[\"Get the file name from the directory path\"].binaryData[\"content\"]}}"}]},"options":{}},"typeVersion":1},{"name":"Get the file name from the directory path","type":"n8n-nodes-base.npmBinary","position":[50,-40],"parameters":{"command":["npx","gitlab-extract-file-name"],"argumentsUi":{"path":"=https://git.gitlab.com/api/v4/projects/16215177/packages/pgp/signatures/gpg.key"},"additionalFields":{}},"credentials":{"npmBin":"Npm Bin Credentials"},"typeVersion":"1"},{"name":"Merge Binary Data","type":"n8n-nodes-base.mergeBinaryData","position":[340,290],"pa

In [22]:
# Test prompt
test_prompt = """<|im_start|>system
You are a helpful assistant that generates n8n workflow JSON based on user requirements.<|im_end|>
<|im_start|>user
Create transcription jobs using AWS Transcribe<|im_end|>
<|im_start|>assistant
"""

# Generate
outputs = pipe(
    test_prompt,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.15,
)

print("\n=== Generated Workflow ===")
print(outputs[0]['generated_text'])

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Generated Workflow ===
<|im_start|>system
You are a helpful assistant that generates n8n workflow JSON based on user requirements.<|im_end|>
<|im_start|>user
Create transcription jobs using AWS Transcribe<|im_end|>
<|im_start|>assistant
{"id":"206","name":"Transcription Job creation with AWS","nodes":[{"name":"On clicking 'execute'","type":"n8n-nodes-base.manualTrigger","position":[470,150],"parameters":{},"typeVersion":1},{"name":"AWS Transcribe","type":"n8n-nodes-base.awsTranscribe","position":[950,310],"parameters":{"fileUri":"={{$node[\"Get audio file from S3 bucket\"].json[\"data\"]}}"},"credentials":{"aws":"AWS Credentials"},"retryOnFail":true,"typeVersion":1},{"name":"Merge results","type":"n8n-nodes-base.merge","position":[1330,230],"parameters":{"mode":"combineByPosition"},"typeVersion":1},{"name":"Append to list","type":"n8n-nodes-base.appendToList","position":[1710,230],"parameters":{"options":{},"fieldToMatch":"jobId"},"typeVersion":1},{"name":"Remove duplicate items",

In [17]:
!zip -r qwen-n8n-finetuned-merged.zip {merged_output_dir}
from google.colab import files
files.download('qwen-n8n-finetuned-merged.zip')

  adding: qwen-n8n-finetuned-merged/ (stored 0%)
  adding: qwen-n8n-finetuned-merged/tokenizer_config.json (deflated 60%)
  adding: qwen-n8n-finetuned-merged/generation_config.json (deflated 39%)
  adding: qwen-n8n-finetuned-merged/model.safetensors (deflated 21%)
  adding: qwen-n8n-finetuned-merged/config.json (deflated 74%)
  adding: qwen-n8n-finetuned-merged/tokenizer.json (deflated 81%)
  adding: qwen-n8n-finetuned-merged/chat_template.jinja (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>